In [1]:
import findspark
findspark.init("C:/spark-2.4.8/spark-2.4.8-bin-hadoop2.7")


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_timestamp, to_date


spark = SparkSession.builder \
    .appName("Covid Analysis") \
    .master("local[*]") \
    .getOrCreate()

spark


C:\Users\anitt\anaconda3\envs\spark37\lib\site-packages\numpy\__init__.py:148: UserWarning: mkl-service package failed to import, therefore Intel(R) MKL initialization ensuring its correct out-of-the box operation under condition when Gnu OpenMP had already been loaded by Python process is not assured. Please install mkl-service package, see http://github.com/IntelPython/mkl-service
  from . import _distributor_init


In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("C:/Vaccines_by_California_County.csv")

df.printSchema()
df.show(5)
df.count()


root
 |-- county: string (nullable = true)
 |-- total_doses: integer (nullable = true)
 |-- cumulative_total_doses: integer (nullable = true)
 |-- pfizer_doses: integer (nullable = true)
 |-- cumulative_pfizer_doses: integer (nullable = true)
 |-- moderna_doses: integer (nullable = true)
 |-- cumulative_moderna_doses: integer (nullable = true)
 |-- jj_doses: integer (nullable = true)
 |-- cumulative_jj_doses: integer (nullable = true)
 |-- partially_vaccinated: integer (nullable = true)
 |-- total_partially_vaccinated: integer (nullable = true)
 |-- fully_vaccinated: integer (nullable = true)
 |-- cumulative_fully_vaccinated: integer (nullable = true)
 |-- at_least_one_dose: integer (nullable = true)
 |-- cumulative_at_least_one_dose: integer (nullable = true)
 |-- booster_recip_count: integer (nullable = true)
 |-- bivalent_booster_recip_count: integer (nullable = true)
 |-- cumulative_booster_recip_count: integer (nullable = true)
 |-- cumulative_bivalent_booster_recip_count: integer

54942524

In [4]:
df_clean = df.withColumn("county", trim(col("county")))

df_clean = df_clean.withColumn(
    "date_parsed",
    to_timestamp(col("date"), "MM/dd/yyyy hh:mm:ss a")
).withColumn(
    "date_parsed", to_date(col("date_parsed"))
)

df_clean.select("date", "date_parsed").show(20, truncate=False)


+----------------------+-----------+
|date                  |date_parsed|
+----------------------+-----------+
|09/02/2021 12:00:00 AM|2021-09-02 |
|08/30/2021 12:00:00 AM|2021-08-30 |
|08/31/2021 12:00:00 AM|2021-08-31 |
|09/01/2021 12:00:00 AM|2021-09-01 |
|09/02/2021 12:00:00 AM|2021-09-02 |
|08/30/2021 12:00:00 AM|2021-08-30 |
|08/31/2021 12:00:00 AM|2021-08-31 |
|09/01/2021 12:00:00 AM|2021-09-01 |
|09/02/2021 12:00:00 AM|2021-09-02 |
|08/30/2021 12:00:00 AM|2021-08-30 |
|08/31/2021 12:00:00 AM|2021-08-31 |
|09/01/2021 12:00:00 AM|2021-09-01 |
|09/02/2021 12:00:00 AM|2021-09-02 |
|08/30/2021 12:00:00 AM|2021-08-30 |
|08/31/2021 12:00:00 AM|2021-08-31 |
|09/01/2021 12:00:00 AM|2021-09-01 |
|09/02/2021 12:00:00 AM|2021-09-02 |
|08/30/2021 12:00:00 AM|2021-08-30 |
|08/31/2021 12:00:00 AM|2021-08-31 |
|09/01/2021 12:00:00 AM|2021-09-01 |
+----------------------+-----------+
only showing top 20 rows



In [5]:
from pyspark.sql.functions import min as spark_min, max as spark_max, countDistinct

row_count = df_clean.count()
county_count = df_clean.select("county").distinct().count()
date_range = df_clean.agg(
    spark_min("date_parsed").alias("min_date"),
    spark_max("date_parsed").alias("max_date")
).collect()[0]

print("Number of rows:", row_count)
print("Number of counties:", county_count)
print("Date range:", date_range['min_date'], "to", date_range['max_date'])

Number of rows: 54942524
Number of counties: 58
Date range: 2020-01-05 to 2025-11-10


In [6]:
from pyspark.sql.functions import sum as spark_sum

county_totals = df_clean.groupBy("county").agg(
    spark_sum("total_doses").alias("total_doses_sum"),
    spark_max("cumulative_total_doses").alias("cumulative_total_doses_sum")
)

county_totals.orderBy(col("total_doses_sum").desc()).show(20, truncate=False)


+--------------+---------------+--------------------------+
|county        |total_doses_sum|cumulative_total_doses_sum|
+--------------+---------------+--------------------------+
|Los Angeles   |12953661112    |26492645                  |
|San Diego     |4639105816     |9486419                   |
|Orange        |3992509879     |8427880                   |
|Santa Clara   |3410298840     |6764407                   |
|Alameda       |2984536187     |5848382                   |
|Riverside     |2335786128     |5071441                   |
|Sacramento    |2005500635     |4151449                   |
|Contra Costa  |1975807015     |3965523                   |
|San Bernardino|1932023458     |4122804                   |
|San Francisco |1724105802     |3418902                   |
|San Mateo     |1494760550     |2947670                   |
|Ventura       |1055111703     |2233528                   |
|Fresno        |971864286      |2035496                   |
|Sonoma        |810566931      |1689093 

In [7]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# window: latest date per county
w = Window.partitionBy("county").orderBy(col("date_parsed").desc())

latest_per_county = df_clean.withColumn(
    "rn", row_number().over(w)
).where(col("rn") == 1).drop("rn")

latest_per_county.select(
    "county",
    "date_parsed",
    "county_pop2020",
    "cumulative_at_least_one_dose",
    "cumulative_fully_vaccinated"
).orderBy("county").show(50, truncate=False)

+---------------+-----------+--------------+----------------------------+---------------------------+
|county         |date_parsed|county_pop2020|cumulative_at_least_one_dose|cumulative_fully_vaccinated|
+---------------+-----------+--------------+----------------------------+---------------------------+
|Alameda        |2025-11-10 |1685886       |1564276                     |1408315                    |
|Alpine         |2025-11-10 |1117          |601                         |532                        |
|Amador         |2025-11-10 |38531         |24077                       |22355                      |
|Butte          |2025-11-10 |217769        |132452                      |121650                     |
|Calaveras      |2025-11-10 |44289         |26208                       |24417                      |
|Colusa         |2025-11-10 |22593         |14592                       |13415                      |
|Contra Costa   |2025-11-10 |1160099       |1064982                     |985916   

In [8]:
from pyspark.sql.functions import round as spark_round

coverage = latest_per_county \
    .filter(col("county_pop2020").isNotNull() & (col("county_pop2020") > 0)) \
    .withColumn(
        "one_dose_coverage_pct",
        spark_round(col("cumulative_at_least_one_dose") / col("county_pop2020") * 100, 2)
    ) \
    .withColumn(
        "fully_vaccinated_coverage_pct",
        spark_round(col("cumulative_fully_vaccinated") / col("county_pop2020") * 100, 2)
    ) \
    .select(
        "county",
        "date_parsed",
        "county_pop2020",
        "cumulative_at_least_one_dose",
        "cumulative_fully_vaccinated",
        "one_dose_coverage_pct",
        "fully_vaccinated_coverage_pct"
    )

coverage.orderBy(col("fully_vaccinated_coverage_pct").desc()).show(20, truncate=False)


+-------------+-----------+--------------+----------------------------+---------------------------+---------------------+-----------------------------+
|county       |date_parsed|county_pop2020|cumulative_at_least_one_dose|cumulative_fully_vaccinated|one_dose_coverage_pct|fully_vaccinated_coverage_pct|
+-------------+-----------+--------------+----------------------------+---------------------------+---------------------+-----------------------------+
|Imperial     |2025-11-10 |191649        |277008                      |181697                     |144.54               |94.81                        |
|Marin        |2025-11-10 |260800        |255655                      |233969                     |98.03                |89.71                        |
|San Mateo    |2025-11-10 |778001        |749430                      |672432                     |96.33                |86.43                        |
|Contra Costa |2025-11-10 |1160099       |1064982                     |985916           

In [9]:
daily_trend = df_clean.groupBy("date_parsed").agg(
    spark_sum("total_doses").alias("total_doses_day")
).orderBy("date_parsed")

daily_trend.show(30, truncate=False)

+-----------+---------------+
|date_parsed|total_doses_day|
+-----------+---------------+
|2020-01-05 |56             |
|2020-07-27 |168            |
|2020-07-28 |392            |
|2020-07-29 |672            |
|2020-07-30 |840            |
|2020-07-31 |336            |
|2020-08-01 |1512           |
|2020-08-02 |336            |
|2020-08-03 |1120           |
|2020-08-04 |1120           |
|2020-08-05 |1176           |
|2020-08-06 |1064           |
|2020-08-07 |1176           |
|2020-08-08 |336            |
|2020-08-09 |224            |
|2020-08-10 |2090           |
|2020-08-11 |1960           |
|2020-08-12 |1848           |
|2020-08-13 |2128           |
|2020-08-14 |1680           |
|2020-08-15 |448            |
|2020-08-16 |224            |
|2020-08-17 |2128           |
|2020-08-18 |1400           |
|2020-08-19 |2408           |
|2020-08-20 |2520           |
|2020-08-21 |3071           |
|2020-08-22 |952            |
|2020-08-23 |560            |
|2020-08-24 |2912           |
+---------

In [10]:
import os, shutil

base = "C:/covid_output"

# Create base folder if it doesn't exist
os.makedirs(base, exist_ok=True)

# Remove old result folders if they exist (Spark needs a clean dir)
for sub in ["coverage_per_county", "daily_trend"]:
    path = os.path.join(base, sub)
    shutil.rmtree(path, ignore_errors=True)

In [11]:
import os

output_base = r"C:/Users/anitt/covid_output"
os.makedirs(output_base, exist_ok=True)
output_base


'C:/Users/anitt/covid_output'

In [12]:
coverage_pd = coverage.toPandas()
coverage_pd.to_csv(os.path.join(output_base, "coverage_per_county.csv"), index=False)


ImportError: Pandas >= 0.19.2 must be installed; however, it was not found.

In [13]:
daily_trend_pd = daily_trend.toPandas()
daily_trend_pd.to_csv(os.path.join(output_base, "daily_trend.csv"), index=False)


ImportError: Pandas >= 0.19.2 must be installed; however, it was not found.

In [14]:
import matplotlib.pyplot as plt

coverage_pd = coverage.toPandas().sort_values(
    "fully_vaccinated_coverage_pct", ascending=False
)

plt.figure(figsize=(12,6))
plt.bar(coverage_pd["county"], coverage_pd["fully_vaccinated_coverage_pct"])
plt.xticks(rotation=90)
plt.ylabel("Fully Vaccinated (%)")
plt.title("Fully Vaccinated Coverage per County")
plt.tight_layout()
plt.show()


ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.7 from "C:\Users\anitt\anaconda3\envs\spark37\python.exe"
  * The NumPy version is: "1.21.5"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed: The specified module could not be found.


In [ ]:
daily_trend_pd = daily_trend.toPandas()

plt.figure(figsize=(12,6))
plt.plot(daily_trend_pd["date_parsed"], daily_trend_pd["total_doses_day"])
plt.xlabel("Date")
plt.ylabel("Total Doses per Day")
plt.title("Daily Vaccination Trend in California")
plt.tight_layout()
plt.show()


In [ ]:
# --- Cumulative Vaccination Trend in California (One Cell) ---

from pyspark.sql.functions import sum as spark_sum
import matplotlib.pyplot as plt

# 1. Aggregate cumulative total doses by date (California-wide)
cumulative_trend = df_clean.groupBy("date_parsed").agg(
    spark_max("cumulative_total_doses").alias("cumulative_doses")
).orderBy("date_parsed")

# 2. Convert to Pandas for visualization
cumulative_trend_pd = cumulative_trend.toPandas()

# 3. Plot cumulative vaccination trend
plt.figure(figsize=(10, 5))
plt.plot(
    cumulative_trend_pd["date_parsed"],
    cumulative_trend_pd["cumulative_doses"]
)
plt.title("Cumulative COVID-19 Vaccination Trend in California")
plt.xlabel("Date")
plt.ylabel("Cumulative Doses Administered")
plt.tight_layout()
plt.show()
